# Reproducing Plitt, Kaganovsky et al. (2023) from dandiset 001710

Paper: Plitt, Kaganovsky, Südhof & Giocomo, "Hippocampal place code plasticity in CA1 requires postsynaptic membrane fusion", bioRxiv doi:10.1101/2023.11.20.567978, version 1 (21 November 2023). The claims file (`claims/001710/10.1101-2023.11.20.567978.yaml`) was written from that preprint's full text: main text, Figure 1 to 7 and S1 to S9 legends, Methods.

Dataset: DANDI:001710, version `0.260213.0329`. The paper (a preprint) cites no dataset version. Two versions are published, `0.260212.2225` and `0.260213.0329`; their asset lists (paths and sizes) are identical (checked below), so the most recent published version is pinned, as the protocol asks.

Outside inputs (the claims file says the paper tracked cells across sessions with custom code): the authors' across-session ROI matches, Figshare doi:10.6084/m9.figshare.31286365 version 5 (`roi_aligners.zip`), and the session-to-aligner index in the authors' code, github.com/GiocomoLab/Plitt_Kaganovsky_Stx3_CA1 at commit `f2ab24d` (`mouse_metadata.py`). Both are named in the dandiset's description. They map ROI indices the dataset holds; they bring no measurements (`with_labels`). Link claim: `link-roi-alignment`.

Activity signal (an assumption): The files hold dF/F, raw fluorescence and neuropil, but not the deconvolved "activity rates" the preprint's Methods describe. The authors' code for these files uses the deposited dF/F as the activity signal in every analysis; this notebook does the same. Every imaging outcome rests on this assumption.

Claims tested and blocked (outcomes are in the summary at the end):

| claim | status here |
|---|---|
| fig1c-novelty-preference, figs1c-, figs1e-, figs1f- (x2) | blocked: freely moving behaviour not in the dataset |
| fig5h-, fig5i- (extinction) | blocked: days 7 and 8 not in the dataset |
| figs4-morph-task, figs5-ca3-dg-place-fields | blocked: sessions not in the dataset |
| fig3bc-, fig3d-, fig3e- (model) | blocked: simulation, not specified by the paper |
| fig2ab-example-cell-counts | tested |
| fig1f-, fig1g-, vr-reward-location-learning, figs2f-, figs2g-, figs2i- | tested (behaviour) |
| fig2cd- (x2), figs3a-, figs3b-, figs3c- | tested |
| fig2gh-, figs3eg- (across days) | tested, depend on the link claim |
| fig3f-, fig3g-, fig3h-, fig3i- (single fields) | tested |
| fig4e- (x3), fig4-block6-, fig4e-novel-increase-, fig4d-, figs6b-, figs6e- | tested |
| figs7b-, fig5d- (x3) | tested |
| fig6c- (x2), figs8h-, figs8d-, fig6d-, fig6e- | tested |
| figs9a-, figs9b-, fig7e- (x3), fig7h- (x3) | tested |

In [ ]:
import warnings, json, re
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import stx3
from stx3 import mixed, fmt_mixed, ttest2, holm, virus_of, CTRL, CRE
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 40)
print("dandiset", stx3.DANDISET, "version", stx3.VERSION)

In [ ]:
A = stx3.list_assets()
A_old = stx3.list_assets("0.260212.2225")
same = {(a["path"], a["size"]) for a in A} == {(a["path"], a["size"]) for a in A_old}
print(len(A), "assets; identical paths and sizes in 0.260212.2225:", same)
assets = pd.DataFrame(A)
print(assets.groupby("group")["mouse"].nunique().to_string())
print(assets.pivot_table(index="mouse", columns="file_day", values="scan", aggfunc="count").fillna(0).astype(int).to_string())

## Survey of every file's metadata (no data read)

In [ ]:
M = stx3.read_all_metadata([a["asset_id"] for a in A])
meta = pd.concat([assets, pd.DataFrame(M)], axis=1)
meta["n_frames"] = meta["dff_shape"].map(lambda s: s[0])
meta["n_cells"] = meta["dff_shape"].map(lambda s: s[1])
print("imaging rate (Hz) files by group:", meta.groupby(["group", meta["imaging_rate"].round(4)]).size().to_dict())
print("dF/F rate equals the imaging-plane rate in every file:", bool((meta["dff_rate"].round(4) == meta["imaging_rate"].round(4)).all()))
print("files at 15.5078 Hz:", meta[meta["imaging_rate"].round(4) == 15.5078]["path"].str.split("/").str[0].value_counts().to_dict())
print("locations:", meta["location"].unique(), "mux:", meta.groupby("group")["mux"].unique().to_dict())
print("multiplexed files: n_rois=-1 where the file has no single segmentation;", int((meta.n_rois < 0).sum()))
print("file day == metadata day:", (meta["file_day"] == meta["meta_day"]).all())
dense = ~meta.mux
print("frames of dF/F == frames of 2P-aligned behaviour (control and Cre files):", (meta.loc[dense, "n_frames"] == meta.loc[dense, "n_aligned"]).all(),
      "; ROI count == dF/F columns:", (meta.loc[~meta.mux, "n_rois"] == meta.loc[~meta.mux, "n_cells"]).all())
print("novel arm constant per mouse:", (meta.groupby("mouse")["novel_arm"].nunique() == 1).all())
print(meta.groupby("group")["subject_description"].unique().map(lambda x: x[0][:120]).to_string())
dup = meta[meta.group.isin(["Ctrl", "Cre"])].groupby(["n_frames", "n_cells"]).size()
print("files sharing frame and cell counts (possible duplicates):", int((dup > 1).sum()))
print(meta[meta.group.isin(["Ctrl", "Cre"])].pivot_table(index="mouse", columns="file_day", values="n_cells", aggfunc="sum").to_string())

## Reduce every control and Cre day to what the claims use

Each day's file (two files for the days split in two, Ctrl-2 file day 3 and Ctrl-4 file day 1, merged on the ROIs the authors matched between them) is streamed once and reduced in `stx3.session_summary`: trial x bin x cell mean dF/F, behaviour per trial and bin at full resolution, 1000 within-trial shuffles per trial type (place fields and recomputed spatial information) and HSEs. This takes about 20 minutes with four processes.

In [ ]:
G = stx3.day_groups(A)
S = stx3.summarize_all(G, workers=4)
print(len(S), "mouse-days reduced")

## Data check

In [ ]:
keys = sorted(S)
mice = sorted({k[0] for k in keys}, key=lambda m: (m[:3], int(m.split("-")[1])))
print("mice:", mice)
rows = []
for (m, fd), s in sorted(S.items()):
    LR = s["LR"]; nov = s["novel_arm"]
    rows.append(dict(mouse=m, virus=virus_of(m), day=fd + stx3.DAY_OFFSET, cells=s["n_cells"], scans=s["n_scans"],
                     trials=len(LR), fam=int((LR == -nov).sum()), nov=int((LR == nov).sum()),
                     blocks=int(s["block"].max()) + 1, nov_first_block=int(s["block"][LR == nov].min()) + 1 if (LR == nov).any() else -1,
                     full_res_trials=len(s["full"]["LR"]), lr_match=bool(np.all(s["full"]["LR"] == LR)) if len(s["full"]["LR"]) == len(LR) else False))
days = pd.DataFrame(rows)
print(days.pivot_table(index="mouse", columns="day", values="cells").to_string())
print(days.pivot_table(index="mouse", columns="day", values="nov").to_string())
print("days with fewer than 6 blocks:", days[days.blocks < 6][["mouse", "day", "blocks", "nov_first_block", "nov"]].to_string(index=False))
print("days per mouse:", days.groupby("mouse").size().to_dict())
print("first block with novel trials, days 1-5:", days[days.day <= 5]["nov_first_block"].value_counts().to_dict(),
      "; day 6:", days[days.day == 6]["nov_first_block"].value_counts().to_dict())
print("full-resolution trials agree with the trial table (count and arm):", days["lr_match"].all())
print(days[~days["lr_match"]][["mouse", "day", "trials", "full_res_trials"]].to_string())
print("blocks per day:", days["blocks"].value_counts().to_dict())
pos_all = np.concatenate([s["full"]["reward_pos"] for s in S.values()])
R_all = np.concatenate([s["R"].reshape(-1) for s in list(S.values())[:5]])
print("dF/F (5 days sample): range %.2f to %.2f, median %.3f" % (np.nanmin(R_all), np.nanmax(R_all), np.nanmedian(R_all)))
spd = np.concatenate([s["full"]["speed"].ravel() for s in S.values()])
print("binned speed (cm/s): 1st, 50th, 99th percentile", np.nanpercentile(spd, [1, 50, 99]).round(1))
dur = np.concatenate([s["full"]["dur"] for s in S.values()]); dist = np.concatenate([s["full"]["dist"] for s in S.values()])
print("trial duration on the track (s): median %.1f; integral of the speed series over a trial: median %.0f (track 300 cm), so the speed is in cm/s, not the 10 cm/s its unit says" % (np.median(dur), np.median(dist)))
# reward zone fronts from where rewards were given (the paper gives no coordinates)
rew = {arm: np.concatenate([s["full"]["reward_pos"][s["full"]["LR"] == arm] for s in S.values()]) for arm in (-1, 1)}
FRONT = {arm: float(np.nanpercentile(rew[arm], 1)) for arm in (-1, 1)}
FB = {arm: int(round(FRONT[arm] - stx3.POS_MIN)) for arm in (-1, 1)}
ANT = {arm: np.arange(FB[arm] - 3, FB[arm]) for arm in (-1, 1)}
for arm, name in ((-1, "left"), (1, "right")):
    print(f"{name} arm: reward positions 1st/50th/99th pct {np.nanpercentile(rew[arm], [1, 50, 99]).round(2)} (spline units, 10 cm); "
          f"zone front {FRONT[arm]:.2f} -> anticipatory bins {ANT[arm].tolist()} (cm {ANT[arm][0]*10}-{(ANT[arm][-1]+1)*10})")

Checks of the description against the data. Control and Cre files are imaged at 15.46 Hz, as the Methods say, or 15.51 Hz (counts above); the helper's frame-based windows (8 frames for 500 ms, 12 for 750 ms, 15 for 1 s) use 15.46 Hz, a 0.3% difference. File `dayN` is the paper's day N+1: Ctrl-3 file day 4 holds 1208 cells and Cre-3 file day 0 holds 936, the paper's Ctrl-3 day 5 and Cre-3 day 1 (Fig 2A-B legends; see claim fig2ab below). Novel trials appear in the last block of days 1 to 5 (block 6 on 78 of 80 days; the two other days have 4 or 5 blocks) and from block 1 on day 6, as in the Methods. Position runs over the 30 bins of the 300 cm track. The speed series is labelled 10 cm/s, but its integral over a trial is about 290, the track's 300 cm, so it is in cm/s: the data are trusted and this is recorded as a finding. The lick series at imaging resolution says it must not be used for comparisons between groups, so licks are taken from the full-resolution non-consummatory lick series. The paper gives no coordinates for the reward zones; the zone front on each arm is taken as the 1st percentile of the positions where rewards were given.

### Clock check

Split-half reliability of the familiar-trial maps with dF/F shifted against the position trace, for one control and one Cre day. A shared clock gives a peak at zero shift.

In [ ]:
for m, fd in (("Ctrl-1", 2), ("Cre-1", 2)):
    aid = [a["asset_id"] for a in A if a.get("mouse") == m and a.get("file_day") == fd][0]
    print(m, "day", fd + 1); print(stx3.clock_check(aid).round(3).to_string(index=False))

The reliability is highest at zero shift for Ctrl-1 (0.962). For Cre-1 it is highest at -8 frames (-0.52 s, 0.956 against 0.953 at zero). In both it falls at larger shifts either way. This is consistent with a shared clock, within the slow calcium response; it was checked on these two days only.

## Link claim: link-roi-alignment

The across-day claims use the authors' ROI matches (Figshare v5). Check that every matched index exists in the dataset's files, how many cells are tracked over all six days, and that matched ROIs sit at the same place in the field of view: after removing each pair of sessions' median displacement, matched centroids should lie within a few pixels, unlike random pairs.

In [ ]:
RI = stx3.ravel_index()
TRACK = {}
link_rows = []
for m in CTRL + CRE:
    al = stx3.load_aligner(m)
    inds = [RI[(m, fd, 0)] for fd in range(6)]
    cm = stx3.common_rois(al, inds)               # (6, n tracked), ROI index in each day's first scan
    cols = np.zeros_like(cm)
    ok_idx = True
    resid, rand = [], []
    rng = np.random.default_rng(0)
    for d in range(6):
        s = S[(m, d)]
        n0 = int(meta[(meta.mouse == m) & (meta.file_day == d) & (meta.scan == 0)]["n_cells"].iloc[0])
        ok_idx &= bool(cm[d].max() < n0)
        if s["n_scans"] == 1:
            cols[d] = cm[d]
        else:   # split day: position of the ROI in the merged session (ROIs present in both scans)
            mp = stx3.split_mapping(m, G[(m, d)])[0]
            lut = {r: j for j, r in enumerate(mp)}
            cols[d] = [lut.get(r, -1) for r in cm[d]]
    keep = (cols >= 0).all(0)
    cols = cols[:, keep]
    TRACK[m] = cols
    c0 = S[(m, 0)]["centroid"][cols[0]]
    for d in range(1, 6):
        cd = S[(m, d)]["centroid"][cols[d]]
        shift = np.median(cd - c0, axis=0)
        resid.append(np.linalg.norm(cd - c0 - shift, axis=1))
        rp = S[(m, d)]["centroid"][rng.integers(0, S[(m, d)]["n_cells"], len(c0))]
        rand.append(np.linalg.norm(rp - c0 - shift, axis=1))
    resid = np.concatenate(resid); rand = np.concatenate(rand)
    link_rows.append(dict(mouse=m, tracked_6_days=int(keep.sum()), dropped_split=int((~keep).sum()),
                          cells_day1=S[(m, 0)]["n_cells"], indices_valid=ok_idx,
                          median_resid_px=float(np.median(resid)), frac_within_10px=float(np.mean(resid < 10)),
                          random_median_px=float(np.median(rand))))
link = pd.DataFrame(link_rows)
print(link.round(2).to_string(index=False))
print("all indices valid:", link.indices_valid.all(), "; tracked cells per mouse: median", link.tracked_6_days.median(),
      "range", link.tracked_6_days.min(), "-", link.tracked_6_days.max())
print("matched pairs within 10 px after the median shift: %.3f (over mice, mean); random pairs median distance %.0f px"
      % (link.frac_within_10px.mean(), link.random_median_px.median()))

**Result: the link holds.** Every matched ROI index exists in the dataset's files. After removing each day pair's median displacement, all matched centroids lie within 10 px (median residual 0.5 to 1.6 px per mouse), against a median of 247 px for random pairs. Between 70 and 562 cells per mouse are tracked over all six days. Ctrl-4 loses 32 tracked cells because they are not among the ROIs matched between the two files of its split day 2.

## Behaviour (Fig 1F-G, S2)

Per trial, from the full-resolution series: time, non-consummatory licks and time-weighted mean speed in each 10 cm bin. Anticipatory lick rate (Methods, VR behavioral analyses) is the lick rate in the 30 cm before the zone front of the trial's arm: reading (i) licks over time in those 30 cm, (ii) the mean of the three bins' rates. All trials of the type are used. The Cre group is Cre-1 to Cre-7: reading (b) of the claims file, Cre-1 to Cre-8, cannot be run, as Cre-8 is not in the dataset.

In [ ]:
def trial_table():
    rows = []
    for (m, fd), s in sorted(S.items()):
        f = s["full"]; nov = s["novel_arm"]
        n = min(len(f["LR"]), len(s["LR"]))
        for i in range(n):
            arm = int(f["LR"][i]); ab = ANT[arm]
            T = f["time"][i]; L = f["licks"][i]; V = f["speed"][i]
            with np.errstate(invalid="ignore", divide="ignore"):
                rates = L[ab] / T[ab]
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, trial=i, block=int(f["block"][i]) + 1, arm=arm,
                             ttype="nov" if arm == nov else "fam",
                             lick_i=L[ab].sum() / T[ab].sum() if T[ab].sum() > 0 else np.nan,
                             lick_ii=np.nanmean(rates),
                             spd_binned=np.nanmean(V), spd_time=f["dist"][i] / f["dur"][i] if f["dur"][i] > 0 else np.nan))
    return pd.DataFrame(rows)
TT = trial_table()
# "block 6" = the block in which novel trials first appear (block 6 on 94 of 96 days; see the data check)
nb = TT[TT.ttype == "nov"].groupby(["mouse", "day"]).block.min().rename("test_block")
TT = TT.join(nb, on=["mouse", "day"])
TT["b6"] = TT.block == TT.test_block
print(TT.groupby(["virus", "ttype"]).size().to_string())
print(TT[["lick_i", "lick_ii", "spd_binned", "spd_time"]].describe().round(2).to_string())

### fig1f-anticipatory-licking-familiar

> On familiar arm trials, control mice licked at a higher rate than Cre mice as they approached the reward zone, across the six days.

In [ ]:
def lick_anova(ttype):
    out = {}
    for rd in ("lick_i", "lick_ii"):
        d = TT[TT.ttype == ttype].groupby(["mouse", "virus", "day"], as_index=False)[rd].mean()
        mm = mixed(d, rd)
        means = d.groupby(["virus", "day"])[rd].mean().unstack().round(2)
        print(f"reading ({rd[-2:].strip('_')}): {fmt_mixed(mm)}")
        print(means.to_string())
        print("  per mouse, mean over days:", d.groupby("mouse")[rd].mean().round(2).to_dict())
        out[rd] = (mm, means)
    return out
LF = lick_anova("fam")

**Result: reproduced.** On familiar trials, control mice lick more in the 30 cm before the zone than Cre mice under both rate readings: virus F(1,14)=21.0, p=0.0004 for (i) and F(1,14)=21.2, p=0.0004 for (ii). Control means are 2.9 to 3.8 licks/s on each day, Cre means 1.2 to 2.7. Only Cre-1 to Cre-7 are in the dataset, so the paper's 8-Cre reading (df 15) cannot be run.

### fig1g-anticipatory-licking-novel

> On novel arm trials, control mice licked at a higher rate than Cre mice as they approached the reward zone.

In [ ]:
LN = lick_anova("nov")

**Result: reproduced.** On novel trials control mice lick more: virus F(1,14)=25.0, p=0.0002 for (i) and F(1,14)=25.6, p=0.0002 for (ii).

### vr-reward-location-learning

> Both control and Cre mice learned the context-dependent reward locations, licking more and running slower only in the bins just before the reward zones, and anticipatory licking on familiar trials changed over days.

Day 6, per mouse: binned lick rate and speed for each arm, divided by the mouse's overall mean over all bins and trials of the day; across-animal means per group and arm. The Methods do not say how the "average running speed" is taken; readings: (i) time-weighted mean speed, (ii) mean of the binned speeds.

In [ ]:
res_vr = {}
for v in ("ctrl", "cre"):
    for arm, name in ((-1, "left"), (1, "right")):
        L_m, V_m, V2_m = [], [], []
        for m in (CTRL if v == "ctrl" else CRE):
            f = S[(m, 5)]["full"]
            tot_rate = f["licks"].sum() / f["time"].sum()
            tot_spd = np.nansum(f["speed"] * f["time"]) / f["time"].sum()      # (i) time-weighted mean speed
            tot_spd2 = np.nanmean(f["speed"])                                     # (ii) mean of the binned speeds
            sel = f["LR"] == arm
            with np.errstate(invalid="ignore", divide="ignore"):
                lr = f["licks"][sel].sum(0) / f["time"][sel].sum(0)
            L_m.append(lr / tot_rate); V_m.append(np.nanmean(f["speed"][sel], 0) / tot_spd)
            V2_m.append(np.nanmean(f["speed"][sel], 0) / tot_spd2)
        Lm, Vm, V2m = np.nanmean(L_m, 0), np.nanmean(V_m, 0), np.nanmean(V2_m, 0)
        ab = ANT[arm]; zone = np.arange(FB[arm], FB[arm] + 3)
        outside = np.setdiff1d(np.arange(30), np.r_[ab, zone])
        ant_l = Lm[ab].mean(); ant_v = Vm[ab].mean(); ant_v2 = V2m[ab].mean()
        holds = bool(ant_l > Lm[outside].max() and ant_v < 1)
        holds2 = bool(ant_l > Lm[outside].max() and ant_v2 < 1)
        res_vr[(v, name)] = (holds, holds2)
        print(f"{v} {name}: norm lick in 30 cm before zone {ant_l:.2f}, max outside {Lm[outside].max():.2f} (bin {outside[np.argmax(Lm[outside])]}); "
              f"norm speed there (i) time-weighted mean {ant_v:.2f}, (ii) bin mean {ant_v2:.2f}; pattern holds: (i) {holds}, (ii) {holds2}")
        print("   norm lick by bin:", np.round(Lm, 2).tolist())
        print("   norm speed by bin:", np.round(Vm, 2).tolist())
for rd in ("lick_i", "lick_ii"):
    mm, means = LF[rd]
    dmean = means.mean(0)
    print(f"fig1f ANOVA, reading {rd}: day F={mm['day']['F']:.3g} p={mm['day']['p']:.3g}; mean day 1 {dmean[1]:.2f}, day 6 {dmean[6]:.2f}")

**Result: partially reproduced.** Under speed reading (ii), the day-6 spatial pattern holds in both groups on both arms. Normalized lick rate in the 30 cm before the zone is 1.69 to 3.50, above every bin outside the zone, where the maximum is 0.84 to 1.65. Normalized speed there is 0.47 to 0.57. Under reading (i), the Cre speed ratio is 1.10, so the pattern holds for control only. The day effect of the familiar-trial licking ANOVA is significant (p=0.007 and p=0.0006). But the day-6 mean (2.67 and 2.38 licks/s) is below the day-1 mean (3.22 and 3.17), and the criterion asks for day 6 above day 1. So the day part fails.

### figs2f-novelty-slowing-familiar

> When the novel arm block began, control mice slowed down more than Cre mice on familiar arm trials, most clearly on day 1.

Normalized speed = log10 of the mean speed on the block 6 trials over the mean speed on the 10 trials before block 6. On two days (data check) the session has 4 or 5 blocks and the novel trials are in the last one; that block is used as block 6. Readings: trials (a) first two familiar trials of block 6, (b) all familiar trials of block 6; speed (i) mean of the binned speeds, (ii) distance over time.

In [ ]:
def slowing(ttype):
    out = {}
    for tr in ("a", "b"):
        for sp in ("spd_binned", "spd_time"):
            rows = []
            for (m, day), d in TT[TT.day <= 5].groupby(["mouse", "day"]):
                d = d.sort_values("trial").reset_index(drop=True)
                first6 = np.flatnonzero(d.b6.values)
                if len(first6) == 0:
                    continue
                pos6 = first6[0]
                base = d.iloc[max(0, pos6 - 10):pos6][sp].mean()
                b6 = d[d.b6 & (d.ttype == ttype)]
                b6 = b6.iloc[:2] if tr == "a" else b6
                rows.append(dict(mouse=m, virus=virus_of(m), day=day, ns=np.log10(b6[sp].mean() / base)))
            df = pd.DataFrame(rows)
            mm = mixed(df, "ns")
            ph = []
            for day in range(1, 6):
                t = ttest2(df[(df.day == day) & (df.virus == "ctrl")].ns, df[(df.day == day) & (df.virus == "cre")].ns)
                ph.append(t)
            adj = holm([t["p"] for t in ph]); adjw = holm([t["p_welch"] for t in ph])
            print(f"reading ({tr},{'i' if sp == 'spd_binned' else 'ii'}): {fmt_mixed(mm)}")
            print("   group means by day:", df.groupby(["virus", "day"]).ns.mean().unstack().round(3).to_dict("index"))
            print("   posthoc ctrl vs cre per day: t=", [round(t["t"], 2) for t in ph], "Holm p=", adj.round(3).tolist(), "Welch Holm p=", adjw.round(3).tolist())
            out[(tr, sp)] = (mm, df, ph, adj, adjw)
    return out
SF = slowing("fam")

**Result: reproduced (reading a,i).** With the first two familiar trials of block 6 and the binned speed, control mice slow more than Cre mice: virus F(1,14)=4.89, p=0.044 (paper F(1,14)=4.82, p=0.045). Control means are -0.08 to -0.14 log units, Cre means -0.03 to -0.08. On day 1, Holm p=0.007 (Student) and 0.006 (Welch), t=-3.98 (paper t=4.43, p=0.007). Reading (a,ii) has a virus effect (p=0.006) but no day-1 effect (Holm p=0.108). Readings (b,i) and (b,ii), all block-6 familiar trials, show no effect (p=0.56, 0.27).

### figs2g-novel-arm-slowing-no-difference

> Control and Cre mice slowed similarly on novel arm trajectories.

Main: first two novel trials of block 6, speed reading (i); the others are variants.

In [ ]:
SN = slowing("nov")

**Result: reproduced.** On the first two novel trials of block 6 (binned speed), the virus main effect is F(1,14)=1.11, p=0.309 (paper F(1,14)=1.12, p=0.308). The variant (a,ii) gives p=0.048.

### figs2i-reward-approach-speed-cv

> Cre mice have more variable running speed during reward approach (higher coefficient of variation) than control mice, on familiar and novel trials.

In [ ]:
CVR = {}
for ttype in ("fam", "nov"):
    rows = []
    for (m, fd), s in S.items():
        f = s["full"]; nov = s["novel_arm"]
        arm = nov if ttype == "nov" else -nov
        V = f["speed"][f["LR"] == arm]
        if len(V) < 2:
            continue
        cv = np.nanstd(V, 0, ddof=1) / np.nanmean(V, 0)
        rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, cv=np.nanmean(cv[ANT[arm]])))
    df = pd.DataFrame(rows); mm = mixed(df, "cv"); CVR[ttype] = mm
    print(ttype, fmt_mixed(mm)); print(df.groupby(["virus", "day"]).cv.mean().unstack().round(3).to_string())

**Result: reproduced.** The speed CV in the 30 cm before the zone is higher in Cre mice on familiar trials (virus F(1,14)=8.33, p=0.012) and on novel trials (F(1,14)=6.32, p=0.025). On day 6 the Cre mean is 0.57 and the control mean 0.29 (familiar).

## Imaging: common definitions

Familiar trials are those on the arm that is not the novel arm; novel trials those on the novel arm. Place cells are the deposited per-arm place-cell masks of each file (`place_cell_info`; they equal shuffle p < 0.05 of the deposited spatial information, checked below); the place cells recomputed here from the same 1000 shuffles are a variant. Per-mouse values are on the paper's days 1 to 6.

In [ ]:
def tidx(s, tt):
    return np.flatnonzero(s["LR"] == (s["novel_arm"] if tt == "nov" else -s["novel_arm"]))

def pcmask(s, tt, src="dep"):
    return s["dep_pc"][tt]["masks"].astype(bool) if src == "dep" else (s["si_p"][tt] < 0.05)

chk = []
for (m, fd), s in S.items():
    for tt in ("fam", "nov"):
        d = s["dep_pc"][tt]
        chk.append(dict(mouse=m, day=fd + 1, tt=tt, dep_eq_p=np.mean(d["masks"].astype(bool) == (d["p"] < 0.05)),
                        frac_dep=d["masks"].mean(), frac_re=(s["si_p"][tt] < 0.05).mean(),
                        agree=np.mean(d["masks"].astype(bool) == (s["si_p"][tt] < 0.05))))
chk = pd.DataFrame(chk)
print("deposited mask == deposited p < 0.05: mean agreement %.4f, min %.4f" % (chk.dep_eq_p.mean(), chk.dep_eq_p.min()))
print("place-cell fraction, deposited vs recomputed (mean over mouse-days):")
print(chk.groupby("tt")[["frac_dep", "frac_re", "agree"]].mean().round(3).to_string())

def block_test(s):
    """Index of the first trial of the block with the first novel trial, and that block's trials."""
    nidx = tidx(s, "nov")
    b = s["block"][nidx[0]]
    tr = np.flatnonzero(s["block"] == b)
    return tr[0], tr

### fig2ab-example-cell-counts

> The example sessions of Fig 2A-B hold 1208 cells and 120 familiar trials (Ctrl-3, day 5), 1568 cells and 20 novel trials (Ctrl-3, day 1), 1087 cells and 17 familiar trials (Cre-3, day 5) and 936 cells and 17 novel trials (Cre-3, day 1).

In [ ]:
ex = [("Ctrl-3", 5, "fam", 1208, 120), ("Ctrl-3", 1, "nov", 1568, 20), ("Cre-3", 5, "fam", 1087, 17), ("Cre-3", 1, "nov", 936, 17)]
FIG2AB = []
for m, day, tt, nc, ntr in ex:
    s = S[(m, day - 1)]
    got_c, got_t = s["n_cells"], len(tidx(s, tt))
    FIG2AB += [got_c, got_t]
    print(f"{m} day {day} {tt}: cells {got_c} (paper {nc}, within 10%: {abs(got_c - nc) <= 0.1 * nc}); "
          f"trials {got_t} (paper {ntr}, within 10%: {abs(got_t - ntr) <= 0.1 * ntr})")
s = S[("Cre-3", 4)]
print("Cre-3 day 5 trials by type:", {tt: len(tidx(s, tt)) for tt in ("fam", "nov")}, "; block-6 familiar trials:", int(np.sum(s["block"][tidx(s, "fam")] == 5)))

**Result: partially reproduced.** Seven of the eight counts match exactly: the four cell counts (1208, 1568, 1087, 936) and the trial counts for Ctrl-3 (120 familiar, 20 novel) and Cre-3 day 1 (17 novel). Cre-3 day 5 has 82 familiar trials, not 17; that day also has 11 novel trials and 7 familiar trials in block 6.

### fig2cd-decoding-no-group-difference

> Controlling for the number of cells, position decoding error did not differ between control and Cre mice on familiar or novel trials.

Decoder input as in the authors' code (dF/F x 10, shifted to be positive); Poisson log likelihood with continuous rates, uniform prior, leave-one-trial-out; 50 draws of 128 cells with replacement per mouse, day and trial type; error in cm over the visited bins.

In [ ]:
rng_dec = np.random.default_rng(2)
dec_rows = []
for (m, fd), s in sorted(S.items()):
    for tt in ("fam", "nov"):
        idx = tidx(s, tt)
        dec_rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, tt=tt,
                             err=stx3.decode_loo(s["R"][idx], s["occ"][idx], rng_dec)))
DEC = pd.DataFrame(dec_rows)
DECM = {}
for tt in ("fam", "nov"):
    d = DEC[DEC.tt == tt]; DECM[tt] = mixed(d, "err")
    print(tt, fmt_mixed(DECM[tt]))
    print(d.groupby(["virus", "day"]).err.mean().unstack().round(1).to_string())
print("decoding error range (cm):", DEC.err.min().round(1), "-", DEC.err.max().round(1), "; chance for a uniform guess is about 100 cm")

**Result: reproduced.** Decoding error does not differ between groups: virus F(1,14)=0.995, p=0.335 on familiar trials and F(1,14)=0.257, p=0.62 on novel trials. Errors range from 6 to 121 cm per mouse and day.

### fig2cd-decoding-improves-over-days

> Decoding error fell as the mice gained experience over days, on familiar and novel trials, with day 1 worse than every later day.

In [ ]:
from scipy import stats
DAYS_PH = {}
for tt in ("fam", "nov"):
    w = DEC[DEC.tt == tt].pivot(index="mouse", columns="day", values="err")
    pairs = [(i, j) for i in range(1, 7) for j in range(i + 1, 7)]
    res = [stats.ttest_rel(w[i], w[j]) for i, j in pairs]
    adj = holm([r.pvalue for r in res])
    d1 = {j: (round(float(res[k].statistic), 2), float(adj[k])) for k, (i, j) in enumerate(pairs) if i == 1}
    DAYS_PH[tt] = d1
    print(tt, "day effect F=%.3g p=%.3g" % (DECM[tt]["day"]["F"], DECM[tt]["day"]["p"]))
    print("   day 1 vs day j (paired t, Holm over 15 pairs):", {j: (t, f"{p:.3g}") for j, (t, p) in d1.items()})
    wc = w.loc[[m for m in w.index if m.startswith("Ctrl")]]
    print("   control only, day 1 vs day 2: t=%.2f p=%.3g (uncorrected)" % tuple(stats.ttest_rel(wc[1], wc[2])))

**Result: partially reproduced.** The day effect is significant on familiar trials (F(5,70)=6.97, p=2.5e-5) and on novel trials (F(5,70)=8.93, p=1.3e-6). On novel trials, day 1 error is above days 2, 3, 4 and 6 (Holm p 0.002 to 2e-5) but not day 5 (Holm p=0.066). On familiar trials, none of the five day-1 contrasts survives Holm correction (p 0.12 to 0.29), though all have t > 2.4.

### figs3a-spatial-information-fraction-no-difference

> The fraction of cells with significant spatial information did not differ between control and Cre mice, on familiar or novel trials.

In [ ]:
S3A = {}
for src in ("dep", "re"):
    rows = [dict(mouse=m, virus=virus_of(m), day=fd + 1, tt=tt, frac=pcmask(s, tt, src).mean())
            for (m, fd), s in S.items() for tt in ("fam", "nov")]
    d = pd.DataFrame(rows)
    for tt in ("fam", "nov"):
        mm = mixed(d[d.tt == tt], "frac"); S3A[(src, tt)] = mm
        print(f"place cells {src}, {tt}: {fmt_mixed(mm)}; means ctrl {d[(d.tt == tt) & (d.virus == 'ctrl')].frac.mean():.3f}, cre {d[(d.tt == tt) & (d.virus == 'cre')].frac.mean():.3f}")

**Result: reproduced.** With the deposited place-cell masks there is no virus effect on the place-cell fraction: F(1,14)=0.182, p=0.676 (familiar) and F(1,14)=0.118, p=0.736 (novel). With the recomputed masks (variant), the novel fraction differs (p=0.036). The fractions are high: 0.88 to 0.89 familiar and 0.66 to 0.68 novel.

### figs3b-within-day-stability-no-difference

> Within-day stability of spatial representations, the mean trial by trial population vector correlation, was similar between control and Cre mice.

Readings: correlation (a) flattened cells x bins, (b) mean over bins of the per-bin correlation (the authors' `calc_pv_corr`); cells (i) all, (ii) place cells of that trial type and day.

In [ ]:
rows = []
for (m, fd), s in S.items():
    for tt in ("fam", "nov"):
        idx = tidx(s, tt)
        for cells in ("i", "ii"):
            cm = np.ones(s["n_cells"], bool) if cells == "i" else pcmask(s, tt)
            for rd in ("a", "b"):
                rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, tt=tt, reading=rd + "," + cells,
                                 pv=stx3.pairwise_pv(s["R"][idx][:, :, cm], rd)))
PVW = pd.DataFrame(rows)
S3B = {}
for (rdg, tt), d in PVW.groupby(["reading", "tt"]):
    mm = mixed(d, "pv"); S3B[(rdg, tt)] = mm
    print(f"reading ({rdg}) {tt}: {fmt_mixed(mm)}; ctrl {d[d.virus == 'ctrl'].pv.mean():.3f}, cre {d[d.virus == 'cre'].pv.mean():.3f}")

**Result: reproduced.** Under all four readings and both trial types, the virus main effect has p >= 0.897. Group means are 0.39 to 0.45 in both groups.

### figs3c-familiar-novel-remapping

> Place codes were largely uncorrelated between the familiar and novel arms (global remapping) in both groups, and the familiar to novel population vector correlation did not differ between control and Cre mice.

In [ ]:
rows = []
for (m, fd), s in S.items():
    fa = np.nanmean(s["R"][tidx(s, "fam")], 0); no = np.nanmean(s["R"][tidx(s, "nov")], 0)
    for cells in ("i", "ii"):
        cm = np.ones(s["n_cells"], bool) if cells == "i" else (pcmask(s, "fam") | pcmask(s, "nov"))
        for rd in ("a", "b"):
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, reading=rd + "," + cells,
                             fn=stx3.pv_corr(fa[:, cm], no[:, cm], rd)))
FN = pd.DataFrame(rows)
S3C = {}
for rdg, d in FN.groupby("reading"):
    mm = mixed(d, "fn"); S3C[rdg] = mm
    within = PVW[PVW.reading == rdg].groupby("virus").pv.mean()
    print(f"reading ({rdg}): {fmt_mixed(mm)}; familiar-novel PV corr ctrl {d[d.virus == 'ctrl'].fn.mean():.3f}, cre {d[d.virus == 'cre'].fn.mean():.3f}; "
          f"within-type trial x trial ctrl {within['ctrl']:.3f}, cre {within['cre']:.3f}")
print("(for reading ii the cells are the union of familiar and novel place cells, as the paper does for sequences across trial types)")

**Result: reproduced, resting on an assumption.** The readings disagree, and reading (b), the per-bin correlation of the authors' code (`calc_pv_corr`), is adopted. Under (b) the virus effect has p=0.314 (cells i) and p=0.316 (ii). Under the flattened reading (a), Cre mice have a higher familiar-to-novel correlation (p=0.018 and 0.020; 0.32 against 0.25). The familiar-to-novel correlation, 0.30 (control) and 0.33 (Cre) under (b,i), is lower than the within-type trial-by-trial correlation, 0.40 in both groups. It is not near zero, as the stem is shared.

### fig2gh-across-day-pv-correlation-no-difference

> The across-day population vector correlation of place cells tracked over all six days did not differ between control and Cre mice, for familiar or novel trials.

Cells tracked over the six days (link claim). Readings: cells (a) all tracked, (b) tracked cells that are place cells of the trial type on at least one day; PV correlation readings a, b as in figs3b.

In [ ]:
def day_maps(m, tt):
    cols = TRACK[m]
    return np.stack([np.nanmean(S[(m, d)]["R"][tidx(S[(m, d)], tt)][:, :, cols[d]], 0) for d in range(6)])

def tracked_pc(m, tt):
    cols = TRACK[m]
    return np.any([pcmask(S[(m, d)], tt)[cols[d]] for d in range(6)], axis=0)

rows = []
for m in CTRL + CRE:
    for tt in ("fam", "nov"):
        DM = day_maps(m, tt)
        for cells in ("a", "b"):
            cm = np.ones(DM.shape[2], bool) if cells == "a" else tracked_pc(m, tt)
            for rd in ("a", "b"):
                C = np.array([[stx3.pv_corr(DM[i][:, cm], DM[j][:, cm], rd) for j in range(6)] for i in range(6)])
                rows.append(dict(mouse=m, virus=virus_of(m), tt=tt, reading=cells + "," + rd,
                                 pv=C[~np.eye(6, dtype=bool)].mean(), n=int(cm.sum())))
AD = pd.DataFrame(rows)
G2GH = {}
for (rdg, tt), d in AD.groupby(["reading", "tt"]):
    t = ttest2(d[d.virus == "ctrl"].pv, d[d.virus == "cre"].pv); G2GH[(rdg, tt)] = t
    print(f"cells,pv ({rdg}) {tt}: ctrl {d[d.virus == 'ctrl'].pv.mean():.3f}, cre {d[d.virus == 'cre'].pv.mean():.3f}; "
          f"t={t['t']:.3f} p={t['p']:.3g} (Welch p={t['p_welch']:.3g}); cells median {int(d.n.median())}")

**Result: reproduced.** Under every reading and both trial types, the across-day PV correlation does not differ between groups: p=0.30 to 0.56 (Welch 0.29 to 0.58). Means are 0.52 to 0.58 (control) and 0.49 to 0.55 (Cre).

### figs3eg-across-day-decoding-no-difference

> Position decoders trained on one day and tested on other days did not differ in accuracy between control and Cre mice.

All tracked cells; the decoder is fit on the mean map of all trials of one day and tested on each trial of every other day. Readings: per-mouse value (a) mean of the off-diagonal entries, (b) with the diagonal filled by the within-day leave-one-trial-out error (all tracked cells, no subsampling).

In [ ]:
rows = []
rng_x = np.random.default_rng(3)
for m in CTRL + CRE:
    cols = TRACK[m]
    for tt in ("fam", "nov"):
        Rs = [S[(m, d)]["R"][tidx(S[(m, d)], tt)][:, :, cols[d]] for d in range(6)]
        Os = [S[(m, d)]["occ"][tidx(S[(m, d)], tt)] for d in range(6)]
        E = np.array([[stx3.decode_cross(Rs[i], Rs[j], Os[j]) if i != j else np.nan for j in range(6)] for i in range(6)])
        off = np.nanmean(E)
        for d in range(6):
            E[d, d] = stx3.decode_loo(Rs[d], Os[d], rng_x, n_cells=None, n_iter=1) if Rs[d].shape[2] else np.nan
        rows.append(dict(mouse=m, virus=virus_of(m), tt=tt, a=off, b=np.nanmean(E)))
XD = pd.DataFrame(rows)
S3EG = {}
for tt in ("fam", "nov"):
    d = XD[XD.tt == tt]
    for rd in ("a", "b"):
        t = ttest2(d[d.virus == "ctrl"][rd], d[d.virus == "cre"][rd]); S3EG[(rd, tt)] = t
        print(f"{tt} reading {rd}: ctrl {d[d.virus == 'ctrl'][rd].mean():.1f} cm, cre {d[d.virus == 'cre'][rd].mean():.1f} cm; t={t['t']:.3f} p={t['p']:.3g} (Welch p={t['p_welch']:.3g})")

**Result: reproduced.** The across-day decoding error does not differ between groups under either reading: p=0.63 to 0.66. Control is 22 to 27 cm, Cre 24 to 29 cm.

## Single place fields (Fig 3F-I)

Fields from 1000 within-trial shuffles of each trial type's trial-averaged map (helper `find_fields`): contiguous bins above the 95th percentile of the null, at least 20 cm. Readings: fields (a) all candidates, (b) only those with a formation lap; width averaged (x) over fields, (y) over cells (each cell's mean width); cells for fields per cell (a) cells with at least one field, (b) place cells of that type and day (cells with no field count 0).

In [ ]:
FLD = []
for (m, fd), s in S.items():
    for tt, flds in s["fields"].items():
        for (c, lo, hi, form) in flds:
            FLD.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, tt=tt, cell=c, width=(hi - lo) * 10.0, form=form >= 0))
FLD = pd.DataFrame(FLD)
print(FLD.groupby(["virus", "tt"]).agg(fields=("width", "size"), with_formation=("form", "mean"), width_mean=("width", "mean")).round(3).to_string())
print("width range (cm):", FLD.width.min(), "-", FLD.width.max())

def width_table(tt, fsel, avg):
    d = FLD[(FLD.tt == tt) & (FLD.form | (fsel == "a"))]
    if avg == "x":
        return d.groupby(["mouse", "virus", "day"], as_index=False).width.mean()
    return d.groupby(["mouse", "virus", "day", "cell"], as_index=False).width.mean().groupby(["mouse", "virus", "day"], as_index=False).width.mean()

def nfield_table(tt, fsel, csel):
    rows = []
    for (m, fd), s in S.items():
        d = FLD[(FLD.mouse == m) & (FLD.day == fd + 1) & (FLD.tt == tt) & (FLD.form | (fsel == "a"))]
        cnt = np.bincount(d.cell.values.astype(int), minlength=s["n_cells"])
        sel = cnt > 0 if csel == "a" else pcmask(s, tt)
        rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, nf=cnt[sel].mean() if sel.any() else np.nan))
    return pd.DataFrame(rows)

def field_claim(tt, kind):
    out = {}
    for r1 in ("a", "b"):
        for r2 in (("x", "y") if kind == "width" else ("a", "b")):
            d = width_table(tt, r1, r2) if kind == "width" else nfield_table(tt, r1, r2)
            v = "width" if kind == "width" else "nf"
            mm = mixed(d[d.day <= 5], v)
            t6 = ttest2(d[(d.day == 6) & (d.virus == "ctrl")][v], d[(d.day == 6) & (d.virus == "cre")][v])
            out[(r1, r2)] = (mm, t6, d.groupby("virus")[v].mean())
            print(f"fields {r1}, {'average' if kind == 'width' else 'cells'} {r2}: days 1-5 {fmt_mixed(mm)}")
            print(f"     day 6 ctrl vs cre t={t6['t']:.2f} p={t6['p']:.3g} (Welch p={t6['p_welch']:.3g}); means ctrl {d[d.virus == 'ctrl'][v].mean():.3f}, cre {d[d.virus == 'cre'][v].mean():.3f}")
    return out

### fig3f-narrower-fields-cre-familiar

> On familiar trials, place fields of Cre mice are narrower than those of control mice.

In [ ]:
F3F = field_claim("fam", "width")

**Result: not reproduced.** Familiar-trial fields of Cre mice are narrower on average (all days, all candidate fields: 63 against 68 cm). But no reading has a significant virus effect over days 1 to 5 (p=0.089 to 0.214) or on day 6 (p=0.060 to 0.209).

### fig3g-narrower-fields-cre-novel

> On novel trials, place fields of Cre mice are narrower than those of control mice.

In [ ]:
F3G = field_claim("nov", "width")

**Result: partially reproduced.** On novel trials Cre fields are narrower. With all candidate fields, day 6 differs: t=2.49, p=0.026 (Welch 0.050) for the field average, and t=2.54, p=0.024 for the cell average. The days 1 to 5 virus effect is not significant under any reading (p=0.051 to 0.082). With formation-lap fields only, neither test is significant.

### fig3h-more-fields-per-cell-cre-familiar

> On familiar trials, place cells of Cre mice have more place fields per cell than those of control mice.

In [ ]:
F3H = field_claim("fam", "nf")

**Result: reproduced.** On familiar trials Cre cells have more fields per cell under all four readings. Over days 1 to 5 the virus p is 0.0009 to 0.043. On day 6, p is 9e-5 to 0.041 (Welch up to 0.051). With all fields and cells with a field, the means are 1.57 (Cre) and 1.39 (control).

### fig3i-more-fields-per-cell-cre-novel

> On novel trials, place cells of Cre mice have more place fields per cell than those of control mice.

In [ ]:
F3I = field_claim("nov", "nf")

**Result: reproduced.** On novel trials Cre cells have more fields per cell under three of four readings, with days 1 to 5 virus p=0.002 to 0.008 and day 6 p=2e-5 to 0.0002. The fourth reading (formation-lap fields, place cells) gives p=0.071 and 0.056.

## Block 6 population activity (Fig 4, S6)

Each cell's binned dF/F on a block trial is divided by the cell's mean over the bins of the 10 trials before the block (as `baseline_avg_act` in the authors' code); the population value is log10 of the mean over cells; the mouse-day value is its mean over bins and block trials of the type (Methods, Novel arm population activity rate). With dF/F, a cell whose baseline mean is at or below 0 has no ratio; such cells are dropped and counted. Variants: (v1) the code's `population` normalisation (mean over cells of the block activity over the mean over cells of the baseline); (v2) Fig 4A legend, log per cell before averaging, over cells and bins where the ratio is positive.

In [ ]:
def block_values(s, block=None):
    """Per trial and bin, normalized population activity for the block (default: first block
    with novel trials). Returns dict with trials, values (main, v1, v2) and dropped cells."""
    if block is None:
        first, tr = block_test(s)
    else:
        tr = np.flatnonzero(s["block"] == block)
        if len(tr) == 0:
            return None
        first = tr[0]
    if first < 10:
        return None
    base_tr = np.arange(first - 10, first)
    base = np.nanmean(s["R"][base_tr], axis=(0, 1))
    ok = base > 0
    X = s["R"][tr][:, :, ok] / base[ok]
    with np.errstate(invalid="ignore", divide="ignore"):
        main = np.log10(np.nanmean(X, axis=2))
        v1 = np.log10(np.nanmean(s["R"][tr], axis=2) / np.nanmean(base))
        lx = np.where(X > 0, np.log10(X), np.nan)
        v2 = np.nanmean(lx, axis=2)
    return dict(trials=tr, main=main, v1=v1, v2=v2, dropped=int((~ok).sum()), n=len(ok), percell=np.nanmean(X, axis=1))

BV = {}
for (m, fd), s in S.items():
    BV[(m, fd)] = block_values(s)
rows = []
for (m, fd), b in BV.items():
    s = S[(m, fd)]
    if b is None:
        continue
    for tt in ("fam", "nov"):
        sel = np.isin(b["trials"], tidx(s, tt))
        for k in ("main", "v1", "v2"):
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, tt=tt, var=k, val=np.nanmean(b[k][sel]),
                             stem_a=np.nanmean(b[k][sel][:, :15]), arm_a=np.nanmean(b[k][sel][:, 15:]),
                             stem_b=np.nanmean(b[k][sel][:, :10]), arm_b=np.nanmean(b[k][sel][:, 10:])))
BLK = pd.DataFrame(rows)
drop = pd.DataFrame([dict(mouse=m, day=fd + 1, dropped=b["dropped"], n=b["n"]) for (m, fd), b in BV.items() if b is not None])
print("cells dropped for baseline <= 0: %d of %d (%.2f%%), per mouse-day max %.1f%%" % (drop.dropped.sum(), drop.n.sum(), 100 * drop.dropped.sum() / drop.n.sum(), 100 * (drop.dropped / drop.n).max()))
print("mouse-days without a block value:", [k for k, b in BV.items() if b is None])
print("value range (main):", BLK[BLK["var"] == "main"].val.min().round(3), BLK[BLK["var"] == "main"].val.max().round(3))

def blk_anova(tt, var="main", days=5):
    d = BLK[(BLK.tt == tt) & (BLK["var"] == var) & (BLK.day <= days)]
    mm = mixed(d, "val")
    ph = [ttest2(d[(d.day == k) & (d.virus == "ctrl")].val, d[(d.day == k) & (d.virus == "cre")].val) for k in range(1, 6)]
    adj = holm([t["p"] for t in ph])
    return mm, ph, adj, d

### fig4e-block6-familiar-activity-lower-cre

> During block 6, population activity on familiar trials, relative to the preceding baseline, is lower in Cre mice than in control mice.

In [ ]:
F4 = {}
for tt in ("fam", "nov"):
    for var in ("main", "v1", "v2"):
        mm, ph, adj, d = blk_anova(tt, var); F4[(tt, var)] = (mm, ph, adj)
        if tt == "fam":
            print(f"{var}: {fmt_mixed(mm)}")
            print("   means by day:", d.groupby(["virus", "day"]).val.mean().unstack().round(3).to_dict("index"))
            print("   posthoc ctrl vs cre (Student t, Holm over 5 days): t=", [round(t["t"], 2) for t in ph], "p=", adj.round(3).tolist())

**Result: reproduced, resting on an assumption (dF/F as activity).** Block-6 familiar-trial activity relative to baseline is lower in Cre mice: virus F(1,14)=22.4, p=0.0003. Daily means are 0.068 to 0.085 (control) and 0.004 to 0.036 (Cre). Posthoc Holm p is 0.013 on day 1 and 0.018 on day 5. The variants agree (v1 p=0.003, v2 p=0.015). No cell had a baseline at or below zero.

### fig4e-block6-novel-activity-lower-cre

> During block 6, population activity on novel trials, relative to the preceding baseline, is lower in Cre mice than in control mice.

In [ ]:
for var in ("main", "v1", "v2"):
    mm, ph, adj = F4[("nov", var)]
    d = BLK[(BLK.tt == "nov") & (BLK["var"] == var) & (BLK.day <= 5)]
    print(f"{var}: {fmt_mixed(mm)}")
    print("   means by day:", d.groupby(["virus", "day"]).val.mean().unstack().round(3).to_dict("index"))
    print("   posthoc ctrl vs cre (Student t, Holm over 5 days): t=", [round(t["t"], 2) for t in ph], "p=", adj.round(3).tolist())

**Result: reproduced, resting on an assumption.** Novel-trial activity is lower in Cre mice: virus F(1,14)=39.9, p=1.9e-5. Posthoc Holm p is 0.004 on day 1 and 0.001 on day 5. The variants agree (v1 p=0.002, v2 p=0.002).

### fig4e-novel-minus-familiar-larger-control

> The difference between novel and familiar trial population activity in block 6 is larger in control mice than in Cre mice.

In [ ]:
F4D = {}
for var in ("main", "v1", "v2"):
    d = BLK[(BLK["var"] == var) & (BLK.day <= 5)].pivot_table(index=["mouse", "virus", "day"], columns="tt", values="val").reset_index()
    d["diff"] = d["nov"] - d["fam"]
    mm = mixed(d, "diff"); F4D[var] = (mm, d.groupby("virus")["diff"].mean())
    print(f"{var}: {fmt_mixed(mm)}; mean novel minus familiar ctrl {d[d.virus == 'ctrl']['diff'].mean():.3f}, cre {d[d.virus == 'cre']['diff'].mean():.3f}")

**Result: reproduced, resting on an assumption.** Novel minus familiar activity is larger in control mice (0.044 against 0.028): virus F(1,14)=5.16, p=0.040. The variants v1 and v2 do not show it (p=0.28, 0.12).

### fig4-block6-activity-increase-pattern

> In block 6 population activity increased above baseline on familiar and novel trials in both groups, more on the arms than on the stem, and more on novel than on familiar trials in both groups.

Per mouse, mean over days 1 to 5 of the main measure; group means. Stem readings: (a) first 150 cm, (b) first 100 cm.

In [ ]:
d = BLK[(BLK["var"] == "main") & (BLK.day <= 5)].groupby(["mouse", "virus", "tt"], as_index=False)[["val", "stem_a", "arm_a", "stem_b", "arm_b"]].mean()
gm = d.groupby(["virus", "tt"])[["val", "stem_a", "arm_a", "stem_b", "arm_b"]].mean()
print(gm.round(3).to_string())
PAT = {}
for v in ("ctrl", "cre"):
    i_ = bool(gm.loc[(v, "fam"), "val"] > 0 and gm.loc[(v, "nov"), "val"] > 0)
    ii_a = bool(all(gm.loc[(v, t), "arm_a"] > gm.loc[(v, t), "stem_a"] for t in ("fam", "nov")))
    ii_b = bool(all(gm.loc[(v, t), "arm_b"] > gm.loc[(v, t), "stem_b"] for t in ("fam", "nov")))
    iii = bool(gm.loc[(v, "nov"), "val"] > gm.loc[(v, "fam"), "val"])
    PAT[v] = (i_, ii_a, ii_b, iii)
    print(f"{v}: (i) both above 0: {i_}; (ii) arm > stem, stem 150 cm: {ii_a}, stem 100 cm: {ii_b}; (iii) novel > familiar: {iii}")
print("per mouse familiar and novel values:", d.pivot(index="mouse", columns="tt", values="val").round(3).to_dict("index"))

**Result: reproduced, resting on an assumption.** In both groups, familiar and novel values are above 0 (control 0.077 and 0.122, Cre 0.023 and 0.051), and novel exceeds familiar. With the stem as the first 150 cm, arm exceeds stem for both types in both groups. With a 100 cm stem, Cre familiar trials do not show it (arm 0.020, stem 0.029).

### fig4e-novel-increase-grows-over-days

> The increase in activity on novel trials did not dissipate over days and grew stronger over the five days.

In [ ]:
for var in ("main", "v1", "v2"):
    mm = F4[("nov", var)][0]
    d = BLK[(BLK.tt == "nov") & (BLK["var"] == var) & (BLK.day <= 5)]
    dm = d.groupby("day").val.mean()
    print(f"{var}: day F({mm['day']['df1']},{mm['day']['df2']})={mm['day']['F']:.3g} p={mm['day']['p']:.3g}; pooled mean day 1 {dm[1]:.3f}, day 5 {dm[5]:.3f}")

**Result: reproduced, resting on an assumption.** The day main effect on novel-trial activity is F(4,56)=4.47, p=0.003 (paper F(4,56)=4.13, p=0.005). The pooled mean rises from 0.078 on day 1 to 0.108 on day 5. Variant v1 gives p=0.067.

### fig4d-activity-increase-spread-across-cells

> On day 5 the larger activity increase of control mice is spread over a large part of the population, roughly half of the cells on novel trials, and is not due to a small population of highly active cells.

Per cell, day 5 block 6 trials of the type: binned ratio to baseline, averaged over bins and trials; per mouse, quantiles of that distribution over cells; across-animal mean per group.

In [ ]:
qs = np.round(np.arange(0.1, 1.0, 0.1), 1)
Q4D = {}
for tt in ("nov", "fam"):
    rows = []
    for m in CTRL + CRE:
        s = S[(m, 4)]; b = BV[(m, 4)]
        sel = np.isin(b["trials"], tidx(s, tt))
        pc = np.nanmean(b["percell"][sel], axis=0)
        rows.append(dict(mouse=m, virus=virus_of(m), **{f"q{q}": np.nanquantile(pc, q) for q in qs}))
    d = pd.DataFrame(rows); g = d.groupby("virus")[[f"q{q}" for q in qs]].mean()
    above = {q: bool(g.loc["ctrl", f"q{q}"] > g.loc["cre", f"q{q}"]) for q in qs}
    Q4D[tt] = (g, above)
    print(tt, "across-animal mean quantile of per-cell ratio to baseline:"); print(g.round(3).to_string())
    print("   control above Cre at deciles:", above)

**Result: reproduced, resting on an assumption.** On day 5 novel trials, the across-animal mean quantile of the per-cell ratio is higher in control than in Cre mice at every decile from 0.2 to 0.9: 1.08 against 0.97 at the median and 2.62 against 1.94 at 0.9. So the shift is not confined to the top tail. Familiar trials show the same at every decile.

### figs6b-block4-activity-no-difference

> In the fourth (familiar) block, normalized population activity did not differ between control and Cre mice.

In [ ]:
rows = []
for (m, fd), s in S.items():
    if fd > 4:
        continue
    b = block_values(s, block=3)
    if b is None:
        continue
    for k in ("main", "v1", "v2"):
        rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, var=k, val=np.nanmean(b[k])))
B4 = pd.DataFrame(rows)
S6B = {}
for var, d in B4.groupby("var"):
    mm = mixed(d, "val"); S6B[var] = mm
    print(f"{var}: {fmt_mixed(mm)}; means ctrl {d[d.virus == 'ctrl'].val.mean():.3f}, cre {d[d.virus == 'cre'].val.mean():.3f}")

**Result: reproduced.** Block-4 normalized activity does not differ between groups: virus F(1,14)=0.118, p=0.736 (paper F(1,14)=0.139, p=0.715). The variants give p=0.95 and 0.55.

### figs6e-familiar-activity-trials-since-novel

> Familiar trial activity in block 6 depends on the number of trials since the last novel trial, in control and in Cre mice.

Per familiar trial of block 6 after the first novel trial, the main population value (mean over bins), grouped by trials since the last novel trial; two-way repeated measures ANOVA (day and level within) per group, complete cases.

In [ ]:
import pingouin as pg
rows = []
for (m, fd), s in S.items():
    b = BV[(m, fd)] if fd < 5 else block_values(s, block=5)
    if b is None:
        continue
    lr = s["LR"][b["trials"]]; nov = s["novel_arm"]
    last = -1
    for k, t in enumerate(b["trials"]):
        if lr[k] == nov:
            last = k; continue
        if last < 0:
            continue
        rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, since=k - last, val=np.nanmean(b["main"][k])))
TS = pd.DataFrame(rows)
S6E = {}
for lv in ("a", "b"):
    for dy in ("i", "ii"):
        d = TS[TS.day <= (5 if dy == "i" else 6)].copy()
        if lv == "a":
            d = d[d.since <= 3]
        else:
            d["since"] = d.since.clip(upper=3)
        g = d.groupby(["mouse", "virus", "day", "since"], as_index=False).val.mean()
        for v in ("ctrl", "cre"):
            gv = g[g.virus == v]
            cnt = gv.groupby("mouse").size()
            full = cnt[cnt == gv.day.nunique() * gv.since.nunique()].index
            gv = gv[gv.mouse.isin(full)]
            try:
                a = pg.rm_anova(data=gv, dv="val", within=["day", "since"], subject="mouse")
                pcol = "p_unc" if "p_unc" in a.columns else "p-unc"
                r = a[a.Source == "since"].iloc[0]
                S6E[(lv, dy, v)] = dict(F=float(r["F"]), p=float(r[pcol]), df=(int(r["ddof1"]), int(r["ddof2"])), n=len(full),
                                        means=gv.groupby("since").val.mean().round(3).to_dict())
            except Exception as e:
                S6E[(lv, dy, v)] = dict(F=np.nan, p=np.nan, df=None, n=len(full), means=gv.groupby("since").val.mean().round(3).to_dict(), err=str(e)[:80])
            print(f"levels ({lv}) days ({dy}) {v}: {S6E[(lv, dy, v)]}")

**Result: reproduced, resting on an assumption.** Under every reading, the trials-since-novel effect is significant in control mice (9 mice, F(2,16)=4.75 to 6.86, p=0.024 to 0.007; paper F(2,16)=6.46, p=0.009). It is also significant in Cre mice (4 mice with complete data, as the paper's df imply; F(2,6)=9.7 to 22.4, p=0.013 to 0.002; paper F(2,6)=15.2, p=0.004). Activity is highest on the trial just after a novel trial and falls with the trials since.

## Reward representation (Fig 5, S7)

Zone fronts as in the data check. A cell's peak is the bin of the maximum of its trial-averaged map on all trials of the type.

In [ ]:
def peaks(s, tt):
    M = np.nanmean(s["R"][tidx(s, tt)], 0)
    return np.nanargmax(np.nan_to_num(M, nan=-np.inf), axis=0)

def arm_of(s, tt):
    return s["novel_arm"] if tt == "nov" else -s["novel_arm"]

### figs7b-reward-overrepresentation

> Control mice allocate a greater proportion of place cells to the locations just before the reward zone than Cre mice, on familiar and novel trials.

In [ ]:
S7B = {}
for src in ("dep", "re"):
    rows = []
    for (m, fd), s in S.items():
        for tt in ("fam", "nov"):
            pcm = pcmask(s, tt, src); fb = FB[arm_of(s, tt)]
            pk = peaks(s, tt)[pcm]
            k = int(((pk >= fb - 5) & (pk < fb)).sum()); n = int(pcm.sum())
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, tt=tt, prop=k / n, lg=stx3.logit_prop(k, n)))
    d = pd.DataFrame(rows)
    for tt in ("fam", "nov"):
        mm = mixed(d[d.tt == tt], "lg"); S7B[(src, tt)] = (mm, d[d.tt == tt].groupby("virus").prop.mean())
        print(f"place cells {src}, {tt}: {fmt_mixed(mm)}; mean proportion ctrl {d[(d.tt == tt) & (d.virus == 'ctrl')].prop.mean():.3f}, cre {d[(d.tt == tt) & (d.virus == 'cre')].prop.mean():.3f}")

**Result: reproduced.** Among deposited place cells, control mice have a larger proportion with peaks in the 50 cm before the zone. On familiar trials this is 0.219 against 0.192 (virus F(1,14)=5.57, p=0.033). On novel trials it is 0.217 against 0.174 (F(1,14)=9.27, p=0.009). With recomputed place cells, the familiar effect is p=0.052.

### fig5d-reward-cells-fewer-in-cre

> Control mice have a larger proportion of reward cells, active before the reward zone on both arms, than Cre mice, and the proportion changes over days.

Readings: window (a) the 50 cm before the front, (b) within 50 cm of the front on either side; denominator (i) all cells, (ii) cells significant on both trial types.

In [ ]:
def in_win(pk, fb, w):
    return (pk >= fb - 5) & (pk < fb) if w == "a" else (pk >= fb - 5) & (pk < fb + 5)

rows = []
for (m, fd), s in S.items():
    both = pcmask(s, "fam") & pcmask(s, "nov")
    pf, pn = peaks(s, "fam"), peaks(s, "nov")
    for w in ("a", "b"):
        rc = both & in_win(pf, FB[arm_of(s, "fam")], w) & in_win(pn, FB[arm_of(s, "nov")], w)
        for den in ("i", "ii"):
            n = s["n_cells"] if den == "i" else int(both.sum())
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, reading=w + "," + den, k=int(rc.sum()), n=n,
                             prop=rc.sum() / n, lg=stx3.logit_prop(int(rc.sum()), n)))
RC = pd.DataFrame(rows)
F5D = {}
for rdg, d in RC.groupby("reading"):
    mm = mixed(d, "lg"); F5D[rdg] = mm
    print(f"reading ({rdg}): {fmt_mixed(mm)}")
    print("   mean proportion by day:", d.groupby(["virus", "day"]).prop.mean().unstack().round(4).to_dict("index"))

**Result: reproduced.** Control mice have more reward cells under all four readings (virus p=0.0004 to 0.038). The proportion changes over days under readings (a,i) and (b,i) (day p=0.016 and 6e-5) but not (a,ii) and (b,ii) (p=0.56 and 0.062).

### fig5d-reward-cells-increase-over-days-control

> The proportion of reward cells increased over days in control mice.

In [ ]:
for rdg, d in RC.groupby("reading"):
    mm = F5D[rdg]; c = d[d.virus == "ctrl"].groupby("day").prop.mean()
    print(f"reading ({rdg}): day F={mm['day']['F']:.3g} p={mm['day']['p']:.3g}; control mean day 1 {c[1]:.4f}, day 6 {c[6]:.4f}, increase {c[6] > c[1]}")

**Result: reproduced (reading a,i).** The day main effect is F(5,70)=3.01, p=0.016. The control mean rises from 0.034 on day 1 to 0.054 on day 6. Reading (b,i) agrees (p=6e-5, 0.097 to 0.169). The day effect is not significant under the denominators of significant cells (p=0.56, 0.062), though the control means still rise.

### fig5d-reward-cells-above-chance-control-only

> Averaged across days, the fraction of reward cells exceeds the level expected from random remapping in control mice but not in Cre mice.

Chance readings: (a) per mouse and day, permute the novel-trial peaks across the cells (significant on both types) whose novel peak is on the arm (bin 15 on, stem and arms of equal length), stem peaks kept, 1000 permutations; (b) analytic: arm peaks uniform over the 15 arm bins, independent between types; stem peaks equal. One-sample t-test of the per-mouse mean over days of observed minus chance.

In [ ]:
rng_ch = np.random.default_rng(5)
rows = []
for (m, fd), s in S.items():
    both = pcmask(s, "fam") & pcmask(s, "nov")
    pf, pn = peaks(s, "fam")[both], peaks(s, "nov")[both]
    fbf, fbn = FB[arm_of(s, "fam")], FB[arm_of(s, "nov")]
    for w in ("a", "b"):
        obs = int((in_win(pf, fbf, w) & in_win(pn, fbn, w)).sum())
        arm = pn >= 15
        perm = []
        for _ in range(1000):
            q = pn.copy(); q[arm] = rng_ch.permutation(pn[arm])
            perm.append((in_win(pf, fbf, w) & in_win(q, fbn, w)).sum())
        ca = float(np.mean(perm))
        winn = np.flatnonzero(in_win(np.arange(30), fbn, w))
        p_arm = np.isin(np.arange(15, 30), winn).mean()
        fw = in_win(pf, fbf, w)
        cb = float(np.sum(np.where(pf[fw] < 15, in_win(pf[fw], fbn, w).astype(float), p_arm)))
        for den in ("i", "ii"):
            n = s["n_cells"] if den == "i" else int(both.sum())
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, reading=w + "," + den, obs=obs / n, chance_a=ca / n, chance_b=cb / n))
CH = pd.DataFrame(rows)
F5C = {}
for rdg, d in CH.groupby("reading"):
    g = d.groupby(["mouse", "virus"], as_index=False)[["obs", "chance_a", "chance_b"]].mean()
    for ch in ("chance_a", "chance_b"):
        for v in ("ctrl", "cre"):
            x = g[g.virus == v]
            t = stats.ttest_1samp(x.obs - x[ch], 0)
            F5C[(rdg, ch, v)] = (float(t.statistic), float(t.pvalue), float((x.obs - x[ch]).mean()))
        print(f"fraction ({rdg}), {ch}: ctrl obs {g[g.virus == 'ctrl'].obs.mean():.4f} vs chance {g[g.virus == 'ctrl'][ch].mean():.4f}: t={F5C[(rdg, ch, 'ctrl')][0]:.2f} p={F5C[(rdg, ch, 'ctrl')][1]:.3g}; "
              f"cre obs {g[g.virus == 'cre'].obs.mean():.4f} vs chance {g[g.virus == 'cre'][ch].mean():.4f}: t={F5C[(rdg, ch, 'cre')][0]:.2f} p={F5C[(rdg, ch, 'cre')][1]:.3g}")

**Result: reproduced (chance reading b).** With the analytic chance (b) and window (a): control mice exceed chance (0.046 against 0.035, t=5.32, p=0.0007); Cre mice do not (0.029 against 0.029, p=0.925). The same holds with denominator (ii). With the permutation chance (a), Cre mice also exceed chance (p=0.002 and 0.002), which the criterion counts as partial. With the wider window (b), the analytic chance exceeds the observed fraction in both groups.

## Place field shifts (Fig 6, S8)

Initial PV shift: spatial cross-correlation of population vectors (lags -290 to 290 cm, PV correlation over the overlapping bins), shift = centre of mass in cm, mean of the upper triangle of the first five novel trials (helper `initial_shift`). Readings: cells (a) all, (b) novel place cells of the day; COM (i) raw cross-correlation, (ii) negative values set to zero.

In [ ]:
rows = []
for (m, fd), s in S.items():
    if fd > 4:
        continue
    Rn = s["R"][tidx(s, "nov")]
    for cells in ("a", "b"):
        cm = np.ones(s["n_cells"], bool) if cells == "a" else pcmask(s, "nov")
        for com in ("i", "ii"):
            rows.append(dict(mouse=m, virus=virus_of(m), day=fd + 1, reading=cells + "," + com,
                             shift=stx3.initial_shift(Rn[:, :, cm], clip_negative=(com == "ii"))))
PVS = pd.DataFrame(rows)
print("initial shift range (cm):", PVS["shift"].min().round(1), PVS["shift"].max().round(1))

def pvs_tests(d):
    mm = mixed(d, "shift")
    ph = [ttest2(d[(d.day == k) & (d.virus == "ctrl")]["shift"], d[(d.day == k) & (d.virus == "cre")]["shift"]) for k in range(1, 6)]
    adj = holm([t["p"] for t in ph])
    one = [(v, k, stats.ttest_1samp(d[(d.day == k) & (d.virus == v)]["shift"], 0)) for v in ("ctrl", "cre") for k in range(1, 6)]
    adj1 = holm([o[2].pvalue for o in one])
    return mm, ph, adj, one, adj1

### fig6c-pv-backward-shift-day1-control

> Control mice show a population-level backward shift of place fields over the first novel arm trials of day 1, larger than in Cre mice; it is not detectable on later days.

In [ ]:
F6C = {}
for rdg, d in PVS.groupby("reading"):
    F6C[rdg] = pvs_tests(d)
    mm, ph, adj, one, adj1 = F6C[rdg]
    print(f"reading ({rdg}): {fmt_mixed(mm)}")
    print("   means by day:", d.groupby(["virus", "day"])["shift"].mean().unstack().round(2).to_dict("index"))
    print("   ctrl vs cre per day: t=", [round(t["t"], 2) for t in ph], "Holm p=", adj.round(3).tolist())
    print("   one-sample vs 0 (Holm over 10):", [(v, k, round(float(o.statistic), 2), round(float(p), 3)) for (v, k, o), p in zip(one, adj1)])

**Result: reproduced (COM reading ii).** With negative cross-correlation set to zero, all cells, the virus main effect is F(1,14)=16.2, p=0.001. On day 1, control is more negative than Cre (-16.5 against 2.9 cm, Holm p=0.019). The control day-1 shift is below 0 (t=-4.50, Holm p=0.020), and control days 2 to 5 are not (Holm p >= 0.81). Place cells only (b,ii) gives the same (day 1 Holm p=0.023, control vs 0 p=0.009). With the raw COM (i), the virus effect holds (p=0.001 for all cells) but the day-1 tests do not (Holm p=0.118 and 0.41).

### fig6c-no-pv-shift-cre

> In Cre mice there is no significant population-wide backward shift within a day.

In [ ]:
for rdg in F6C:
    one, adj1 = F6C[rdg][3], F6C[rdg][4]
    cre = [(k, round(float(o.statistic), 2), round(float(p), 3)) for (v, k, o), p in zip(one, adj1) if v == "cre"]
    print(f"reading ({rdg}) Cre one-sample tests (day, t, Holm p):", cre, "; all p >= 0.05:", all(c[2] >= 0.05 for c in cre))

**Result: reproduced.** No Cre one-sample test against 0 is significant on any day under any reading: Holm p >= 0.88.

### figs8h-across-day-pv-shift

> Across days, novel arm place codes shift backward gradually in control mice but not in Cre mice.

Tracked cells; the PV shift between the day-averaged novel maps of each pair of days; per mouse the mean of the upper triangle. Readings: days (a) 1 to 6, (b) 1 to 5; COM (i), (ii).

In [ ]:
rows = []
for m in CTRL + CRE:
    DM = day_maps(m, "nov")
    for dy, nd in (("a", 6), ("b", 5)):
        for com in ("i", "ii"):
            v = [stx3.xcorr_shift(DM[i], DM[j], com == "ii") for i in range(nd) for j in range(i + 1, nd)]
            rows.append(dict(mouse=m, virus=virus_of(m), reading=dy + "," + com, shift=np.nanmean(v)))
XS = pd.DataFrame(rows)
S8H = {}
for rdg, d in XS.groupby("reading"):
    c, k = d[d.virus == "ctrl"]["shift"], d[d.virus == "cre"]["shift"]
    t = ttest2(c, k); oc = stats.ttest_1samp(c, 0); ok_ = stats.ttest_1samp(k, 0)
    S8H[rdg] = (t, oc, ok_)
    print(f"reading ({rdg}): ctrl {c.mean():.2f} cm (vs 0 t={oc.statistic:.2f} p={oc.pvalue:.3g}), cre {k.mean():.2f} cm (vs 0 t={ok_.statistic:.2f} p={ok_.pvalue:.3g}); "
          f"ctrl vs cre t={t['t']:.2f} p={t['p']:.3g} (Welch p={t['p_welch']:.3g})")

**Result: not reproduced.** Across days, the control novel-arm shift is not below 0 under any reading: means -9.3 to 2.4 cm, p=0.15 to 0.83. Control and Cre do not differ (p=0.065 to 0.31). Cre means are positive, 8.7 to 17.4 cm, with p=0.10 to 0.26 against 0.

### figs8d-pv-shift-not-explained-by-speed

> The day 1 PV shift is not correlated with running speed on the first five novel trials.

In [ ]:
d1 = PVS[(PVS.day == 1) & (PVS.reading == "a,i")].set_index("mouse")["shift"]
sp5 = TT[(TT.day == 1) & (TT.ttype == "nov")].sort_values("trial").groupby("mouse").head(5).groupby("mouse").spd_binned.mean()
j = pd.concat([d1, sp5], axis=1).dropna()
r8d = stats.pearsonr(j["shift"], j["spd_binned"])
print(f"N={len(j)} mice; Pearson r={r8d.statistic:.3f} p={r8d.pvalue:.3g}; speed range {j.spd_binned.min():.1f}-{j.spd_binned.max():.1f} cm/s")
for rdg in ("a,ii", "b,i", "b,ii"):
    dd = pd.concat([PVS[(PVS.day == 1) & (PVS.reading == rdg)].set_index("mouse")["shift"], sp5], axis=1).dropna()
    rr = stats.pearsonr(dd["shift"], dd["spd_binned"]); print(f"   variant PV reading ({rdg}): r={rr.statistic:.3f} p={rr.pvalue:.3g}")

**Result: reproduced.** The day-1 initial PV shift is not correlated with running speed over the first five novel trials: r=-0.109, p=0.689, 16 mice (paper r=-0.317, p=0.231). The other PV readings give r=-0.15 to -0.39, p >= 0.135.

### fig6d-field-com-shift-larger-control

> Single place fields shift backward abruptly after their formation lap on novel trials, more in control than in Cre mice, after controlling for running speed on the formation lap; the difference holds on days 1, 2 and 3.

Novel-trial fields with a formation lap. COM (from frames, unbinned) on each lap, relative to the field's mean COM over laps from the formation lap on, in cm; speed = mean speed through the field on the formation lap. statsmodels MixedLM, shift ~ speed + virus (Cre = 1), random intercept per mouse; Wald test of the virus term. Readings: "trial 1" is (a) the formation lap, (b) the lap after it. Under this measure a field that moves backward after forming has a positive value on the formation lap. Variant (c): the COM on the lap after the formation lap minus the COM on the formation lap (negative = backward), shown because the claims file's sign convention and the paper's "backward shift" point opposite ways under reading (a).

In [ ]:
import statsmodels.formula.api as smf
import warnings as _w
from statsmodels.tools.sm_exceptions import ConvergenceWarning
_w.simplefilter("ignore", ConvergenceWarning)
rows = []
for (m, fd), s in S.items():
    for L in s["field_laps"].get("nov", []):
        c, lo, hi, form = L["field"]
        com = L["com"]; ref = np.nanmean(com[form:])
        rows.append(dict(mouse=m, virus=virus_of(m), cre=int(virus_of(m) == "cre"), day=fd + 1, cell=c,
                         width=(hi - lo) * 10.0, speed=L["speed"][form],
                         sh_a=(com[form] - ref) * 10.0, sh_b=(com[form + 1] - ref) * 10.0 if form + 1 < len(com) else np.nan,
                         sh_c=(com[form + 1] - com[form]) * 10.0 if form + 1 < len(com) else np.nan))
FL = pd.DataFrame(rows)
print("novel fields with a formation lap:", FL.groupby("virus").size().to_dict(), "; by day:", FL.groupby("day").size().to_dict())
F6D = {}
for rd in ("sh_a", "sh_b", "sh_c"):
    for day in range(1, 7):
        d = FL[(FL.day == day)].dropna(subset=[rd, "speed"])
        try:
            fit = smf.mixedlm(f"{rd} ~ speed + cre", d, groups=d["mouse"]).fit()
            b, se = fit.params["cre"], fit.bse["cre"]
            chi2 = (b / se) ** 2; p = float(stats.chi2.sf(chi2, 1))
        except Exception as e:
            b = se = chi2 = p = np.nan
        F6D[(rd, day)] = dict(b=b, chi2=chi2, p=p, ctrl=d[d.cre == 0][rd].mean(), cre=d[d.cre == 1][rd].mean(), n=len(d))
        print(f"reading {rd[-1]}, day {day}: n={len(d)}; mean shift ctrl {F6D[(rd, day)]['ctrl']:.2f} cm, cre {F6D[(rd, day)]['cre']:.2f} cm; "
              f"virus coef (Cre - ctrl) {b:.2f} cm, Wald chi2={chi2:.2f} p={p:.3g}")

**Result: not reproduced.** Under no reading is there a virus effect on day 1. With the formation lap relative to the field mean, Wald chi2=0.05, p=0.815. With the next lap, p=0.984. In variant (c), next lap minus formation lap, p=0.956. Both groups show a small backward step after the formation lap in variant (c): -0.74 cm (control) and -0.68 cm (Cre) on day 1. The only significant virus terms are on day 4 (p=0.036 and 0.032), where Cre fields shift more.

### fig6e-width-speed-correlation

> Place field width correlates with running speed through the field on the formation lap in both groups, and neither the intercept nor the slope differs between control and Cre.

In [ ]:
F6E = {}
for rdg, d in (("a", FL[FL.day == 1]), ("b", FL)):
    d = d.dropna(subset=["speed", "width"])
    fit = smf.mixedlm("width ~ speed * cre", d, groups=d["mouse"]).fit()
    pv = fit.pvalues
    per = {}
    for v in (0, 1):
        dv = d[d.cre == v]
        fv = smf.mixedlm("width ~ speed", dv, groups=dv["mouse"]).fit()
        per[v] = (float(fv.params["speed"]), float(fv.pvalues["speed"]))
    F6E[rdg] = dict(p_cre=float(pv["cre"]), p_int=float(pv["speed:cre"]), per=per, n=len(d))
    print(f"reading {rdg}: n={len(d)}; slope ctrl {per[0][0]:.3f} cm per cm/s (p={per[0][1]:.3g}), cre {per[1][0]:.3f} (p={per[1][1]:.3g}); "
          f"virus term p={pv['cre']:.3g}, speed x virus p={pv['speed:cre']:.3g}")

**Result: reproduced (reading a).** On day 1 novel trials, field width rises with formation-lap speed in both groups: 0.43 cm per cm/s in control (p=9e-78) and 0.46 in Cre (p=7e-45). Neither the virus term (p=0.688) nor the speed by virus term (p=0.409) is significant. Pooling days 1 to 6 (reading b), the speed by virus term is significant (p=0.001).

## HSEs and reactivation (Fig 7, S9)

HSEs per the Methods (helper `hse`): post-reward stopping periods, transients above 20% of the session maximum beginning in the period, coactivity within 8 frames, threshold the 95th percentile of 1000 per-cell circular shuffles. The Methods shuffle "each cell's timeseries" within each period but count only transients that "began after the animal stopped"; whether the null also counts transients that began before the stop and were moved into the period by the shift is open. Readings: S (shift the activity series and recompute onsets, the Methods' words) and O (shift only the onsets that began in the period). Every HSE claim is shown under both; reading labels in the reactivation claims are prefixed S: or O:.

In [ ]:
rows = []
for hk in ("hse", "hse_on"):
    for (m, fd), s in S.items():
        h = s[hk]
        rows.append(dict(mode=hk, mouse=m, virus=virus_of(m), day=fd + 1, periods=h["n_periods"], thr=h["thr"], n_hse=len(h["events"]),
                         frac_trials=h["trial_hse"].mean(),
                         size=np.mean([len(e["cells"]) / s["n_cells"] for e in h["events"]]) if h["events"] else np.nan))
HSALL = pd.DataFrame(rows)
for hk, HS in HSALL.groupby("mode"):
    print("reading", "series (shift activity)" if hk == "hse" else "onsets (shift onsets)")
    print(HS.groupby("virus")[["periods", "thr", "n_hse", "frac_trials", "size"]].mean().round(3).to_string())
    print("   mouse-days without any HSE:", HS[HS.n_hse == 0][["mouse", "day"]].values.tolist())

### figs9a-hse-trial-proportion-no-difference

> The proportion of trials containing a highly synchronous event does not differ between control and Cre mice.

In [ ]:
S9A = {}
for hk, HS in HSALL.groupby("mode"):
    S9A[hk] = mixed(HS, "frac_trials"); print(hk, fmt_mixed(S9A[hk]))
    print(HS.groupby(["virus", "day"]).frac_trials.mean().unstack().round(3).to_string())

**Result: not reproduced, resting on an assumption.** Cre mice have more trials with HSEs under both shuffle readings: S, 0.26 against 0.02, virus F(1,14)=13.1, p=0.003; O, 0.46 against 0.20, F(1,14)=9.3, p=0.009. Under reading S, 19 of the 54 control mouse-days have no HSE.

### figs9b-hse-size-no-difference

> The proportion of the population participating in HSEs does not differ between control and Cre mice.

In [ ]:
S9B = {}
for hk, HS in HSALL.groupby("mode"):
    S9B[hk] = mixed(HS, "size"); print(hk, fmt_mixed(S9B[hk]), "(mouse-days without HSEs have no value; the ANOVA keeps mice with all six days)")
    print(HS.groupby(["virus", "day"])["size"].mean().unstack().round(4).to_string())
    g = HS.groupby(["mouse", "virus"], as_index=False)["size"].mean()
    t = ttest2(g[g.virus == "ctrl"]["size"], g[g.virus == "cre"]["size"])
    print(f"   variant, per-mouse mean over the days with HSEs: ctrl {g[g.virus == 'ctrl']['size'].mean():.4f} (n={g[g.virus == 'ctrl']['size'].notna().sum()}), cre {g[g.virus == 'cre']['size'].mean():.4f}; t={t['t']:.2f} p={t['p']:.3g}")

**Result: reproduced.** The fraction of cells per HSE does not differ between groups. Under reading O, 7 control and 7 Cre mice have HSEs on every day: virus F(1,12)=0.135, p=0.72, means 0.020 and 0.022. Under reading S, only 1 control mouse has HSEs every day, so the ANOVA has 1 control mouse (p=0.985). The per-mouse t-test variant gives p=0.67 (S) and 0.65 (O).

### fig7e-reactivated-cells-more-stable-across-days-control

> In control mice, cells that participated in at least one HSE have higher across-day spatial rate map correlation than cells that did not, on familiar and novel trials.

Tracked cells. Per cell, Pearson correlation of its day-averaged maps; readings: (a) mean over all pairs of days, (b) over consecutive days; reactivated (i) in an HSE on any day, (ii) on the earlier day of the pair. Per mouse and type, mean of reactivated minus mean of non-reactivated; one-sample t-tests, Holm over the four tests (two groups, two types).

In [ ]:
def part_tracked(m, hk):
    cols = TRACK[m]
    P = np.zeros((6, cols.shape[1]), bool)
    for d in range(6):
        ev = S[(m, d)][hk]["events"]
        cells = np.unique(np.concatenate([e["cells"] for e in ev])) if ev else np.array([], int)
        P[d] = np.isin(cols[d], cells)
    return P

def cellcorr(a, b):
    a = np.nan_to_num(a) - np.nan_to_num(a).mean(0); b = np.nan_to_num(b) - np.nan_to_num(b).mean(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (a * b).sum(0) / np.sqrt((a * a).sum(0) * (b * b).sum(0))

rows = []
for hk in ("hse", "hse_on"):
  for m in CTRL + CRE:
    P = part_tracked(m, hk)
    for tt in ("fam", "nov"):
        DM = day_maps(m, tt)
        for pr in ("a", "b"):
            pairs = [(i, j) for i in range(6) for j in range(i + 1, 6)] if pr == "a" else [(i, i + 1) for i in range(5)]
            C = np.array([cellcorr(DM[i], DM[j]) for i, j in pairs])
            for re_ in ("i", "ii"):
                if re_ == "i":
                    r = P.any(0); diff = np.nanmean(C[:, r]) - np.nanmean(C[:, ~r])
                else:
                    diff = np.nanmean([np.nanmean(C[k, P[i]]) - np.nanmean(C[k, ~P[i]]) for k, (i, j) in enumerate(pairs) if P[i].any() and (~P[i]).any()])
                rows.append(dict(mouse=m, virus=virus_of(m), tt=tt, reading=("S" if hk == "hse" else "O") + ":" + pr + "," + re_, diff=diff, n_react=int(P.any(0).sum()), n=P.shape[1]))
RE = pd.DataFrame(rows)
for pre in ("S", "O"):
    print(pre, "tracked cells reactivated on any day:", RE[RE.reading.str.startswith(pre)].drop_duplicates("mouse").set_index("mouse")[["n_react", "n"]].to_dict("index"))
F7E = {}
for rdg, d in RE.groupby("reading"):
    tests = [(v, tt, stats.ttest_1samp(d[(d.virus == v) & (d.tt == tt)]["diff"].dropna(), 0)) for v in ("ctrl", "cre") for tt in ("fam", "nov")]
    adj = holm([t[2].pvalue for t in tests])
    F7E[rdg] = {(v, tt): (float(t.statistic), float(p), float(d[(d.virus == v) & (d.tt == tt)]["diff"].mean())) for (v, tt, t), p in zip(tests, adj)}
    print(f"reading ({rdg}):", {f"{v} {tt}": f"mean {x[2]:.3f}, t={x[0]:.2f}, Holm p={x[1]:.3g}" for (v, tt), x in F7E[rdg].items()})

**Result: not reproduced, resting on an assumption.** In control mice, reactivated tracked cells are not more stable across days under any reading. The differences are -0.067 to 0.005, and none is positive and significant. Under reading O with all day pairs, they are significantly lower (familiar -0.067, Holm p=0.020; novel -0.056, p=0.015).

### fig7e-cre-reactivation-boost-novel-only

> In Cre mice, reactivated cells have a weakly higher across-day correlation on novel trials but not on familiar trials.

In [ ]:
for rdg, x in F7E.items():
    n_, f_ = x[("cre", "nov")], x[("cre", "fam")]
    print(f"reading ({rdg}): Cre novel mean {n_[2]:.3f} t={n_[0]:.2f} Holm p={n_[1]:.3g}; Cre familiar mean {f_[2]:.3f} t={f_[0]:.2f} Holm p={f_[1]:.3g}")

**Result: not reproduced, resting on an assumption.** In Cre mice, reactivated cells are less stable across days on novel trials (differences -0.014 to -0.116; significant and negative under the O readings, Holm p=0.003 to 0.008). On familiar trials they are also less stable (-0.016 to -0.156), not just equal.

### fig7e-reactivation-effect-virus-and-novelty

> The across-day stability gain of reactivated cells is larger in control than in Cre mice and larger on novel than familiar trials, and the novelty effect is larger in control mice.

In [ ]:
F7EV = {}
for rdg, d in RE.groupby("reading"):
    mm = mixed(d, "diff", within="tt")
    w = d.pivot(index=["mouse", "virus"], columns="tt", values="diff").reset_index(); w["nf"] = w["nov"] - w["fam"]
    t = ttest2(w[w.virus == "ctrl"].nf, w[w.virus == "cre"].nf)
    F7EV[rdg] = (mm, t, d.groupby(["virus", "tt"])["diff"].mean())
    print(f"reading ({rdg}): {fmt_mixed(mm, 'tt')}; novel minus familiar ctrl vs cre t={t['t']:.2f} p={t['p']:.3g} (Welch p={t['p_welch']:.3g})")
    print("   means:", d.groupby(["virus", "tt"])["diff"].mean().round(3).to_dict())

**Result: not reproduced, resting on an assumption.** The virus main effect is significant under six of eight readings (p=0.004 to 0.045), with control less negative than Cre. The trial-type main effect is not significant under any reading (p=0.08 to 0.98), and the novelty comparison is not either (p=0.20 to 0.84).

### fig7h-reactivated-cells-higher-trial-correlation-control

> In control mice, cells that participated in HSEs have higher trial by trial rate map correlation than cells that did not, on familiar and novel trials.

Per cell, day and type, mean Pearson correlation of its single-trial maps over trial pairs. Readings: reactivated (i) on that day, (ii) on any day; cells (a) all cells, (b) tracked cells. Reading (a, ii) needs a cell's HSEs on other days, which is defined only for tracked cells, so it equals (b, ii) and is not shown separately.

In [ ]:
TC = {}
for (m, fd), s in S.items():
    for tt in ("fam", "nov"):
        TC[(m, fd, tt)] = stx3.cell_trial_corr(s["R"][tidx(s, tt)])
rows = []
for hk in ("hse", "hse_on"):
  for m in CTRL + CRE:
    P = part_tracked(m, hk); cols = TRACK[m]
    for tt in ("fam", "nov"):
        for rdg in ("a,i", "b,i", "b,ii"):
            diffs = []
            for fd in range(6):
                s = S[(m, fd)]; c = TC[(m, fd, tt)]
                ev = s[hk]["events"]
                part = np.zeros(s["n_cells"], bool)
                if ev:
                    part[np.unique(np.concatenate([e["cells"] for e in ev]))] = True
                if rdg == "a,i":
                    cc, pp = c, part
                elif rdg == "b,i":
                    cc, pp = c[cols[fd]], part[cols[fd]]
                else:
                    cc, pp = c[cols[fd]], P.any(0)
                if pp.any() and (~pp).any():
                    diffs.append(np.nanmean(cc[pp]) - np.nanmean(cc[~pp]))
            rows.append(dict(mouse=m, virus=virus_of(m), tt=tt, reading=("S" if hk == "hse" else "O") + ":" + rdg, diff=np.nanmean(diffs) if diffs else np.nan))
TH = pd.DataFrame(rows)
F7H = {}
for rdg, d in TH.groupby("reading"):
    tests = [(v, tt, stats.ttest_1samp(d[(d.virus == v) & (d.tt == tt)]["diff"].dropna(), 0)) for v in ("ctrl", "cre") for tt in ("fam", "nov")]
    adj = holm([t[2].pvalue for t in tests])
    F7H[rdg] = {(v, tt): (float(t.statistic), float(p), float(d[(d.virus == v) & (d.tt == tt)]["diff"].mean())) for (v, tt, t), p in zip(tests, adj)}
    print(f"reading ({rdg}):", {f"{v} {tt}": f"mean {x[2]:.4f}, t={x[0]:.2f}, Holm p={x[1]:.3g}" for (v, tt), x in F7H[rdg].items()})

**Result: not reproduced, resting on an assumption.** In control mice, reactivated cells do not have a higher trial-by-trial correlation under any reading. The differences are -0.049 to 0.008. Under reading O:b,ii they are significantly lower (familiar -0.049, Holm p=0.017; novel -0.031, p=0.046).

### fig7h-no-trial-correlation-boost-cre

> In Cre mice, reactivated cells do not have higher trial by trial correlation than non-reactivated cells, on either trial type.

In [ ]:
for rdg, x in F7H.items():
    print(f"reading ({rdg}): Cre familiar mean {x[('cre', 'fam')][2]:.4f} Holm p={x[('cre', 'fam')][1]:.3g}; Cre novel mean {x[('cre', 'nov')][2]:.4f} Holm p={x[('cre', 'nov')][1]:.3g}")

**Result: not reproduced, resting on an assumption.** The paper's tests do find an effect in Cre mice, but in the other direction. Reactivated cells have a lower trial-by-trial correlation: differences -0.022 to -0.121, Holm p=0.0006 to 0.044 under five of six readings. Under reading S:b,ii, familiar p=0.115 and novel p=0.033. They are not higher in any reading, which is the paper's statement. But the claim of absence is judged on the tests, which find a difference.

### fig7h-reactivation-effect-virus-and-novelty

> The trial by trial correlation gain of reactivated cells is larger in control than in Cre mice and larger on novel than familiar trials.

In [ ]:
for rdg, d in TH.groupby("reading"):
    mm = mixed(d, "diff", within="tt")
    print(f"reading ({rdg}): {fmt_mixed(mm, 'tt')}; means {d.groupby(['virus', 'tt'])['diff'].mean().round(4).to_dict()}")

**Result: not reproduced, resting on an assumption.** The virus main effect is significant under every reading (p=0.011 to 0.036), with control higher, that is, less negative. The trial-type main effect is not significant under any reading (p=0.05 to 0.74).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| link-roi-alignment | reproduced (the link holds) |
| fig1c-novelty-preference, figs1c-training-occupancy-no-difference, figs1e-open-field-distance-no-difference, figs1f-epm-closed-arm-no-difference, figs1f-epm-distance-no-difference | blocked, data_missing (freely moving behaviour not deposited) |
| fig2ab-example-cell-counts | partially reproduced (7 of 8 counts; Cre-3 day 5 has 82 familiar trials, not 17) |
| fig1f-anticipatory-licking-familiar, fig1g-anticipatory-licking-novel | reproduced |
| vr-reward-location-learning | partially reproduced (spatial pattern holds; familiar licking falls, not rises, from day 1 to day 6) |
| figs2f-novelty-slowing-familiar, figs2g-novel-arm-slowing-no-difference, figs2i-reward-approach-speed-cv | reproduced |
| fig2cd-decoding-no-group-difference | reproduced* |
| fig2cd-decoding-improves-over-days | partially reproduced* |
| figs3a-, figs3b-, figs3c- (S3C on the adopted reading), fig2gh-, figs3eg- | reproduced* |
| figs4-morph-task-context-discrimination, figs5-ca3-dg-place-fields | blocked, data_missing |
| fig3bc-, fig3d-, fig3e- (model) | blocked, methods_underspecified |
| fig3f-narrower-fields-cre-familiar | not reproduced* |
| fig3g-narrower-fields-cre-novel | partially reproduced* |
| fig3h-, fig3i- (more fields per cell) | reproduced* |
| fig4e- (x3), fig4-block6-activity-increase-pattern, fig4e-novel-increase-grows-over-days, fig4d-, figs6b-, figs6e- | reproduced* |
| figs7b-, fig5d- (x3) | reproduced* |
| fig5h-extinction-group-difference, fig5i-extinction-intercept-not-slope | blocked, data_missing (days 7 and 8 not deposited) |
| fig6c-pv-backward-shift-day1-control, fig6c-no-pv-shift-cre, figs8d-, fig6e- | reproduced* |
| figs8h-across-day-pv-shift, fig6d-field-com-shift-larger-control | not reproduced* |
| figs9a-hse-trial-proportion-no-difference | not reproduced* |
| figs9b-hse-size-no-difference | reproduced* |
| fig7e- (x3), fig7h- (x3) | not reproduced* |

\* rests on the assumption that the deposited dF/F stands for the paper's deconvolved activity rates.

How the analysis differs from the paper's:
- Activity is the deposited dF/F, as in the authors' code for these files, not the deconvolved activity rates of the preprint's Methods.
- Cre-8 is not in the dataset, so the 8-Cre readings of the behaviour claims are not run.
- The reward zone fronts are estimated from where rewards were given.
- HSE stopping periods use a speed from the spline position smoothed over 1 s, because the speed series jitters while the mouse is still. The HSE null is run under two readings.
- Several method details the paper leaves open are run under every reading named in the claims file; the results above say which readings match.
- The two days split over two files are merged on the ROIs the authors matched between them.

What was found about the dataset:
- The dataset holds the head-fixed imaging and VR behaviour of Ctrl-1 to Ctrl-9 and Cre-1 to Cre-7 on days 1 to 6, plus seven SparseKO mice that are not in this preprint.
- The freely moving behaviour, the reversal days 7 and 8, the Cre-8 CA2/3 sessions, the dentate gyrus session and the morph task are not deposited.
- The deconvolved activity rates are not deposited.
- The speed series is labelled 10 cm/s but is in cm/s.
- The across-day cell matches are not in the NWB files; they are on Figshare.
- The two published versions have identical files.